# Final Project Notebook

DS 5001 Text as Data


# Metadata

- Full Name: Joseph Kaminetz
- Userid: emv3bc
- GitHub Repo URL: https://github.com/Joekam03/TextProject
- UVA Box URL:

# Overview

The goal of the final project is for you to create a **digital analytical edition** of a corpus using the tools, practices, and perspectives you’ve learning in this course. You will select a corpus that has already been digitized and transcribed, parse that into an F-compliant set of tables, and then generate and visualize the results of a series of fitted models. You will also draw some tentative conclusions regarding the linguistic, cultural, psychological, or historical features represented by your corpus. The point of the exercise is to have you work with a corpus through the entire pipeline from ingestion to interpretation. 

Specifically, you will acquire a collection of long-form texts and perform the following operations:

- **Convert** the collection from their source formats (F0) into a set of tables that conform to the Standard Text Analytic Data Model (F2).
- **Annotate** these tables with statistical and linguistic features using NLP libraries such as NLTK (F3).
- **Produce** a vector representation of the corpus to generate TFIDF values to add to the TOKEN (aka CORPUS) and VOCAB tables (F4).
- **Model** the annotated and vectorized model with tables and features derived from the application of unsupervised methods, including PCA, LDA, and word2vec (F5).
- **Explore** your results using statistical and visual methods.
- **Present** conclusions about patterns observed in the corpus by means of these operations.

When you are finished, you will make the results of your work available in GitHub (for code) and UVA Box (for data). You will submit to Gradescope (via Canvas) a PDF version of a Jupyter notebook that contains the information listed below.

# Some Details

- Please fill out your answers in each task below by editing the markdown cell. 
- Replace text that asks you to insert something with the thing, i.e. replace `(INSERT IMAGE HERE)` with an image element, e.g. `![](image.png)`.
- For URLs, just paste the raw URL directly into the text area. Don't worry about providing link labels using `[label](link)`.
- Please do not alter the structure of the document or cell, i.e. the bulleted lists. 
- You may add explanatory paragraphs below the bulleted lists.
- Please name your tables as they are named in each task below.
- Tasks are indicated by headers with point values in parentheses.

# Raw Data

## Source Description (1)

Provide a brief description of your source material, including its provenance and content. Tell us where you found it and what kind of content it contains.

This corpus contains plaintext national constitution documents from the GitHub repository `marcomorucci/Clustering-Constitutions`, specifically the `constitutions` directory. According to the Github repository, the constitutions were originally sourced from https://www.constituteproject.org/. Each source document is a country constitution identified by country name and year, such as `Afghanistan_2004.txt`. The content is legal and governmental prose: preambles, articles, chapters, sections, rights, duties, institutional rules, and amendment or enforcement provisions. 


## Source Features (1)

Add values for the following items. (Do this for all following bulleted lists.)

- Source URL: https://github.com/marcomorucci/Clustering-Constitutions/tree/master/constitutions
- UVA Box URL:
- Number of raw documents: 192
- Total size of raw documents (e.g. in MB): approximately 26.5 MB / 26,515,008 characters
- File format(s), e.g. XML, plaintext, etc.: plaintext `.txt`


## Source Document Structure (1)

Provide a brief description of the internal structure of each document. That, describe the typical elements found in document and their relation to each other. For example, a corpus of literary texts may be organized into chapters, paragraphs, sentences, and tokens.

The documents are parsed with the OHCO structure `doc -> div1 -> div2 -> unit -> para -> sent -> token`. The parser treats titles and parts as high-level `div1` divisions, chapters and sections as `div2` divisions when present, and articles as the main `unit` level. For constitutions without article headings, numbered provisions are used as the unit fallback. Paragraphs are created from non-heading text blocks, then split into sentences and word tokens. A variable OHCO structure like this was chosen because not every constitution has a defined hierarchical structure. Using a variable OHCO allows our analysis to capture whatever structure is present in the database. Codex helped me to automate the parsing of the data using the explained OHCO structure.


In [23]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import PCA
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.manifold import TSNE as tsne
from gensim.parsing.porter import PorterStemmer
from gensim.models import word2vec

sns.set_theme(style='darkgrid')

# Get the file listing from the GitHub API
api_url = "https://api.github.com/repos/marcomorucci/Clustering-Constitutions/contents/constitutions"
response = requests.get(api_url, headers={"User-Agent": "FinalProject-OHCO"}, timeout=30)
response.raise_for_status()
files = sorted(response.json(), key=lambda x: x["name"])

constitutions = {}
lib_rows = []

for file_info in files:
    if not file_info["name"].endswith(".txt"):
        continue

    source_filename = file_info["name"]
    doc_id = source_filename.removesuffix(".txt")
    match = re.match(r"^(?P<country>.+)_(?P<year>\d{4})$", doc_id)
    country = match.group("country").replace("_", " ") if match else doc_id.replace("_", " ")
    year = int(match.group("year")) if match else np.nan

    raw_url = file_info["download_url"]
    raw_text = requests.get(raw_url, headers={"User-Agent": "FinalProject-OHCO"}, timeout=30).text
    constitutions[doc_id] = raw_text

    clean_lines = [line.strip() for line in raw_text.splitlines() if line.strip() and line.strip() != "Share"]
    lib_rows.append({
        "doc": doc_id,
        "country": country,
        "year": year,
        "source_filename": source_filename,
        "source_url": raw_url,
        "title_line": clean_lines[0] if clean_lines else "",
        "char_len": len(raw_text),
        "line_count": len(clean_lines),
    })

LIB = pd.DataFrame(lib_rows).set_index("doc").sort_index()

print(f"Loaded {len(constitutions)} constitutions")
print(f"Average document length: {LIB['char_len'].mean():,.0f} characters")
LIB.head()


Loaded 192 constitutions
Average document length: 138,099 characters


,country,year,source_filename,source_url,title_line,char_len,line_count
doc,,,,,,,
Afghanistan_2004,Afghanistan,2004,Afghanistan_2004.txt,https://raw.githubusercontent.com/marcomorucci...,Afghanistan 2004,66806,453
Albania_2008,Albania,2008,Albania_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Albania 1998 (rev. 2008),86022,792
Algeria_2008,Algeria,2008,Algeria_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Algeria 1963 (rev. 2008),66590,658
Andorra_1993,Andorra,1993,Andorra_1993.txt,https://raw.githubusercontent.com/marcomorucci...,Andorra 1993,55687,412
Angola_2010,Angola,2010,Angola_2010.txt,https://raw.githubusercontent.com/marcomorucci...,Angola 2010,175148,1393
